# Project 4: Prediction of Medical Analysis Results
## Executive Master « Data Engineering » (MSDE)

### Objective
The goal of this project is to develop a machine learning pipeline to predict the result of medical analyses (Normal, Abnormal, or Inconclusive) based on patient data. This is a multi-class classification problem.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib

import warnings
warnings.filterwarnings('ignore')

### 1. Data Loading

In [ ]:
df = pd.read_csv('healthcare_dataset.csv')
df.head()

### 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='Test Results', data=df)
plt.title('Distribution of Test Results')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Age'], bins=20, kde=True)
plt.title('Age Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Test Results', y='Age', data=df)
plt.title('Age vs Test Results')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(x='Medical Condition', hue='Test Results', data=df)
plt.title('Medical Condition vs Test Results')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Admission Type', hue='Test Results', data=df)
plt.title('Admission Type vs Test Results')
plt.show()

### 3. Data Preprocessing & Feature Engineering

In [ ]:
df['Admission Date'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'])
df['Stay_Duration'] = (df['Discharge Date'] - df['Admission Date']).dt.days

cols_to_drop = ['Name', 'Doctor', 'Hospital', 'Date of Admission', 'Discharge Date']
df = df.drop(columns=cols_to_drop)

le = LabelEncoder()
df['Test Results'] = le.fit_transform(df['Test Results'])

X = df.drop('Test Results', axis=1)
y = df['Test Results']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

### 4. Model Construction (Testing 10 Algorithms)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'AdaBoost': AdaBoostClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
    'LightGBM': LGBMClassifier(verbose=-1),
    'CatBoost': CatBoostClassifier(verbose=0),
    'SVM': SVC(probability=True)
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f"{name} Accuracy: {accuracy_score(y_test, y_pred):.4f}")

### 5. Tuning & Final Model

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10]
}

rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                             ('classifier', RandomForestClassifier(random_state=42))])

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Final Test Accuracy:", accuracy_score(y_test, grid_search.predict(X_test)))